# PLant Disease classifiaction


In [24]:
import os, re, random, numpy as np
from pathlib import Path
from sklearn.metrics import average_precision_score
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip /content/drive/MyDrive/train.zip -d /content/
!unzip /content/drive/MyDrive/val.zip -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/__MACOSX/train/soybean mosaic/._soybean_mosaic_Bing_0253.jpg  
  inflating: /content/train/soybean mosaic/soybean_mosaic_Bing_0078.jpg  
  inflating: /content/__MACOSX/train/soybean mosaic/._soybean_mosaic_Bing_0078.jpg  
  inflating: /content/train/soybean mosaic/soybean_mosaic_Bing_0044.jpg  
  inflating: /content/__MACOSX/train/soybean mosaic/._soybean_mosaic_Bing_0044.jpg  
  inflating: /content/train/soybean mosaic/soybean_mosaic_Bing_0050.jpg  
  inflating: /content/__MACOSX/train/soybean mosaic/._soybean_mosaic_Bing_0050.jpg  
  inflating: /content/train/soybean mosaic/soybean_mosaic_Bing_0118.jpg  
  inflating: /content/__MACOSX/train/soybean mosaic/._soybean_mosaic_Bing_0118.jpg  
  inflating: /content/train/soybean mosaic/soybean_mosaic_Bing_0108.jpg  
  inflating: /content/__MACOSX/train/soybean mosaic/._soybean_mosaic_Bing_0108.jpg  
  inflating: /content/train/soybean mosaic/soybean_mosaic_Google_0053

In [4]:
folders = os.listdir('/content/train')
print(len(folders))
print(folders[:10])

83
['cabbage downy mildew', 'cabbage alternaria leaf spot', 'peach brown rot', 'banana bunchy top', 'apple black rot', 'cabbage black rot', 'tobacco mosaic virus', 'bell pepper powdery mildew', 'grape leaf spot', '.DS_Store']


#Preparing data


separating disease and plant for each folder

In [5]:
MULTI_WORD_PLANTS = {"bell pepper"}

def parse_folders(root_dir):
    root = Path(root_dir)
    folders = sorted([f for f in root.iterdir() if f.is_dir()])

    disease_set, plant_set = set(), set()
    folder_meta = []

    for folder in folders:
        name = folder.name.lower().strip()
        if name == ".ds_store":
            continue

        matched_plant = next((p for p in MULTI_WORD_PLANTS if name.startswith(p + " ")), None)
        if matched_plant:
            disease_name = name[len(matched_plant)+1:]
        else:
            parts = name.split()
            matched_plant = parts[0]
            disease_name = " ".join(parts[1:])

        disease_set.add(disease_name)
        plant_set.add(matched_plant)
        folder_meta.append((folder, matched_plant, disease_name))

    disease2idx = {d: i for i, d in enumerate(sorted(disease_set))}
    plant2idx = {p: i for i, p in enumerate(sorted(plant_set))}

    print(f"Unique diseases : {len(disease2idx)}")
    print(f"Unique plants   : {len(plant2idx)}")

    samples = []
    for folder, plant_name, disease_name in folder_meta:
        d_idx = disease2idx[disease_name]
        p_idx = plant2idx[plant_name]
        exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
        for img_path in folder.iterdir():
            if img_path.suffix in exts:
                samples.append((str(img_path), d_idx, p_idx))

    print(f"Total images: {len(samples)}")
    return samples, disease2idx, plant2idx

Here I make the imput 224x224 for all pictures and randomly apply tranformations to force model to understand the underlying structure

In [6]:
TRAIN_TF = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.25,
        hue=0.05
    ),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3)
    ], p=0.2),

    transforms.ToTensor(),

    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),

    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

VAL_TF = transforms.Compose([
    transforms.Resize(256),          # keep aspect ratio
    transforms.CenterCrop(224),      # deterministic crop
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [7]:
class PlantDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, d_idx, p_idx = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, d_idx, p_idx

In [8]:
train_samples, disease2idx, plant2idx = parse_folders("/content/train")
val_samples, _, _ = parse_folders("/content/val")
train_ds = PlantDataset(train_samples, TRAIN_TF)
val_ds   = PlantDataset(val_samples,   VAL_TF)

Unique diseases : 39
Unique plants   : 32
Total images: 7783
Unique diseases : 39
Unique plants   : 31
Total images: 390


In [9]:
def split_by_plant(samples, held_out_plant, plant2idx):

    idx2plant = {v: k for k, v in plant2idx.items()}

    train_samples = []
    test_samples  = []

    for sample in samples:
        path, d_idx, p_idx = sample
        plant_name = idx2plant[p_idx]

        if plant_name == held_out_plant:
            test_samples.append(sample)
        else:
            train_samples.append(sample)

    print(f"Held-out plant   : '{held_out_plant}'")
    print(f"Train samples    : {len(train_samples)}")
    print(f"Test samples     : {len(test_samples)}")
    return train_samples, test_samples


# ── Pick which plant to hold out ─────────────────────────────
# First see all available plants:
idx2plant = {v: k for k, v in plant2idx.items()}
print("Available plants:")
for idx, name in sorted(idx2plant.items()):
    count = sum(1 for s in train_samples if s[2] == idx)
    print(f"  {name:20s} {count} images")

Available plants:
  apple                480 images
  banana               256 images
  basil                58 images
  bean                 150 images
  bell pepper          116 images
  blueberry            70 images
  broccoli             76 images
  cabbage              226 images
  cauliflower          35 images
  celery               53 images
  cherry               109 images
  citrus               420 images
  coffee               216 images
  corn                 486 images
  cucumber             373 images
  garlic               95 images
  ginger               61 images
  grape                427 images
  lettuce              105 images
  maple                90 images
  peach                330 images
  plum                 96 images
  potato               190 images
  raspberry            14 images
  rice                 49 images
  soybean              567 images
  squash               169 images
  strawberry           42 images
  tobacco              77 images
  tomato 

In [10]:
# Choose a plant with enough images to test on
HELD_OUT_PLANT = "cabbage"   # ← change to any plant from the list above

# Combine train + val samples first for maximum data
all_samples = train_samples + val_samples

train_split, test_split = split_by_plant(all_samples, HELD_OUT_PLANT, plant2idx)

# Create loaders
train_ds_new = PlantDataset(train_split, TRAIN_TF)
test_ds_new  = PlantDataset(test_split,  VAL_TF)

# Weighted sampler for train split
from collections import Counter
from torch.utils.data import WeightedRandomSampler

disease_counts_new = Counter([s[1] for s in train_split])
weights = [1.0 / disease_counts_new[s[1]] for s in train_split]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds_new, batch_size=32,
                              sampler=sampler, num_workers=8, pin_memory=True)
val_loader = DataLoader(test_ds_new,  batch_size=32,
                              shuffle=False,  num_workers=8, pin_memory=True)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Test batches  : {len(val_loader)}")

Held-out plant   : 'cabbage'
Train samples    : 7943
Test samples     : 230

Train batches : 249
Test batches  : 8


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Creating Model
---

In [11]:
class DiseaseClassifier(nn.Module):
    def __init__(self, num_diseases):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b0", pretrained=True, num_classes=0
        )
        feat_dim = self.backbone.num_features

        for p in self.backbone.parameters():
            p.requires_grad = False

        # Unfreezing last Conv blocks
        for block in self.backbone.blocks[-1:]:
            for p in block.parameters():
                p.requires_grad = True

        self.projector = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        #Disease domain head
        self.disease_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_diseases),
        )

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.projector(feat)

        #Disease prediction
        disease_logits = self.disease_head(feat)

        return disease_logits


Training
---

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [25]:
def compute_map(y_true, y_prob, num_classes):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    y_true_onehot = np.eye(num_classes)[y_true]

    ap_per_class = []

    for c in range(num_classes):
        # skip empty classes
        if np.sum(y_true_onehot[:, c]) == 0:
            continue

        ap = average_precision_score(
            y_true_onehot[:, c],
            y_prob[:, c]
        )
        ap_per_class.append(ap)

    return np.mean(ap_per_class)

In [26]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    all_probs = []
    all_labels = []

    for imgs, d_labels, _ in loader: # Unpack all three values, ignore the third one
        imgs = imgs.to(device)
        d_labels = d_labels.to(device)

        optimizer.zero_grad()

        logits = model(imgs)
        loss = criterion(logits, d_labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        correct += (preds == d_labels).sum().item()
        total += d_labels.size(0)

        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.detach().cpu())
        all_labels.append(d_labels.detach().cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    mAP = compute_map(all_labels, all_probs, num_classes=len(disease2idx))

    acc = 100 * correct / total

    print(
        f"[Train] loss={total_loss/len(loader):.4f} | "
        f"acc={acc:.2f}% | mAP={mAP:.4f}"
    )

    return total_loss / len(loader), mAP

In [27]:
@torch.no_grad()
def val_epoch(model, loader, criterion, device, num_classes):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    all_probs = []
    all_labels = []

    for imgs, d_labels, _ in loader: # Unpack all three values, ignore the third one
        imgs = imgs.to(device)
        d_labels = d_labels.to(device)

        logits = model(imgs)
        loss = criterion(logits, d_labels)

        total_loss += loss.item()

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        correct += (preds == d_labels).sum().item()
        total += d_labels.size(0)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(d_labels.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    acc = 100 * correct / total
    mAP = compute_map(all_labels, all_probs, num_classes)

    print(f"[Val] loss={total_loss/len(loader):.4f} | acc={acc:.2f}% | mAP={mAP:.4f}")

    return acc, mAP

In [31]:
!pip install wandb -q

In [32]:
import wandb
wandb.login()


True

In [33]:
wandb.init(
    project="Plant_Disease_Classification",
    name="experiment-1",
    config={
        "learning_rate": 1e-4,
        "epochs": 10,
        "batch_size": 32
    }
)

In [ ]:

# ── Model ───────────────────
num_diseases = len(disease2idx)
num_plants   = len(plant2idx)

model = DiseaseClassifier(num_diseases).to(device)

    # Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"\nTrainable params: {trainable:,} / {total_p:,} "
        f"({100*trainable/total_p:.1f}%)\n")

optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-5, weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=30, eta_min=1e-6
)

criterion = nn.CrossEntropyLoss()

    # ── Training ─────────────────────
best_map = 0
epochs = 30

for epoch in range(1, epochs + 1):
    print(f"\nEpoch {epoch}/{epochs}")

    avg_train_loss,train_mAP = train_epoch(model, train_loader, optimizer, criterion,
        device)
    test_acc, test_map = val_epoch(model, val_loader, criterion, device, num_diseases)
    wandb.log({
    "train/mAP": train_mAP,
    "val/mAP": test_map,
    "epoch": epoch
})
    scheduler.step()

    if test_map > best_map:
        best_map = test_map
        print(f"\u2705 best model (mAP={test_map:.4f})")

print(f"\n✨ Best Val mAP: {best_map:.4f}")
wandb.finish()


Trainable params: 1,515,479 / 4,805,795 (31.5%)


Epoch 1/30
[Train] loss=3.6557 | acc=4.00% | mAP=0.0322


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[Val] loss=3.6517 | acc=2.61% | mAP=0.3163
✅ best model (mAP=0.3163)

Epoch 2/30


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[Train] loss=3.5836 | acc=7.14% | mAP=0.0559


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[Train] loss=2.2288 | acc=46.19% | mAP=0.4351


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[Val] loss=3.3904 | acc=6.52% | mAP=0.4147

Epoch 18/30


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
def predict(image_path, checkpoint_path="/content/drive/MyDrive/best_model.pth"):
    ckpt = torch.load(checkpoint_path, map_location=device)
    disease2idx = ckpt["disease2idx"]
    plant2idx   = ckpt["plant2idx"]
    idx2disease = {v: k for k, v in disease2idx.items()}

    model = DiseaseClassifier(len(disease2idx), len(plant2idx)).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    img = Image.open(image_path).convert("RGB")
    img = VAL_TF(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits, _ = model(img, lambda_=0.0)
        probs     = F.softmax(logits, dim=1)[0]
        top5      = probs.topk(5)

    print(f"\nPredictions for: {image_path}")
    for prob, idx in zip(top5.values, top5.indices):
        print(f"  {idx2disease[idx.item()]:35s}  {100*prob.item():.2f}%")

In [ ]:
predict('/content/val/apple black rot/apple_black_rot_google_0056.jpg')